# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashcodes07/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yashcodes07/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. One row = one URL, snapshotted for a given month (page-month grain), not page-day.
2. Table(s): `warehouse.page_performance_daily` (or whatever your course's actual table
   name is — check the session slides/BQ dataset browser) rolled up to month=2026-03.
3. Time window: single mid-panel month, month=2026-03, chosen to avoid partial-month
   edge effects at the start/end of the full panel.
4. Predict/rank: refresh-worthiness score — proxy = CTR or position improvement potential
   for pages with declining performance (Lane 2: Refresh/Content Opportunity Scoring).
5. Deliberately excluded: page publish_date as a raw feature — excluded because it's a
   near-duplicate of days_since_published (which I do use), and keeping both risks
   redundant/collinear signal without adding information.

In [2]:
import duckdb

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_XXXXXXXXXXXXXXXXXXX')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Confirm actual column names before writing any more queries
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0
""")


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket    | Field(s)                                              | Why |
|-----------|--------------------------------------------------------|-----|
| Feature   | impressions, clicks, avg_position, ctr (rolled up per content, per month) | Observable performance signals at the decision moment |
| Feature   | content_age_days / days_since_last_update (from dim_content) | Static-ish context available before any refresh decision |
| Label/proxy | opportunity_score (derived from decline in impressions/clicks within the month) | This is the thing I'm ranking by, not an input |
| Context   | client_id, content_type, main_intent (from dim_content/dim_clients) | Useful for grouping/filtering, not fed into the score itself |
| Excluded  | fact_content_query_90d fields | Different grain (content × query); out of scope for this notebook |
| Excluded  | ga4_data_start / gsc_data_start as a raw feature | Used only to check availability, not as a signal — including it as a "feature" would leak panel-onboarding artifacts into the score |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
 schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0
""").df()

# print all rows so nothing gets truncated
import pandas as pd
pd.set_option("display.max_rows", None)
schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [4]:
# 1. Grain check: prove one row = one (report_date, client_hash_id, content_hash_id)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
""").df()
print("Rows violating one-row-per-key claim:", len(grain_check))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating one-row-per-key claim: 0


In [5]:
# 2. Row count and date span for the slice
span = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
span

,row_count,earliest_date,latest_date,n_content_items,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [6]:
# 3. Availability — row-level flag, filtered with IS TRUE
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell me:
- **Unbalanced panel:** `dim_clients.gsc_data_start` / `ga4_data_start` differ
  per client, so earlier months for some clients are GSC-only (no GA4
  engagement signal) — a month=2026-03 row means different data completeness
  depending on which client it belongs to.
- **Single-month window:** one month can't separate a real decay trend from
  normal seasonal noise; a real opportunity score would need multiple months.
- **No post-refresh outcomes:** this fact table shows current/past
  performance, never what happens after a page is actually refreshed, so I
  can't validate the proxy against a true outcome from this data alone.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.